# Project 1: The Multimodal Telephone Game

**Module:** Multimodal & Advanced Models | **Team size:** 2 | **Weight:** 20% of your final grade

In this project you build a chain of AI models that passes a message around the loop

**image → Describe → text → Speak → audio → Listen → text → Draw → image → …**

and measure how the meaning drifts. Read the project overview page on the course website for the full requirements, grading, and Game Day rules.

## How this notebook is organized

| Part | What you do | TODOs |
|------|-------------|-------|
| 0 | Setup: install libraries, load models | none |
| A | Build the four stations | TODO 1–4 |
| — | Check your stations with `check_stations()` | none (run it) |
| B | Build the game loop | TODO 5–6 |
| C | Measure the drift on at least 5 images × 5 rounds | TODO 7–8 |
| D | Two extensions | TODO 9 |
| — | Game Day helper and submission | none |

Code marked **PROVIDED** is ready to use; you should read it but do not need to change it. Code marked **TODO** is yours to write.

**Runtime:** in Colab, choose **Runtime → Change runtime type → T4 GPU** before you start.

## Part 0: Setup

Colab already has PyTorch and `transformers`. This cell installs the rest.

In [ ]:
!pip -q install diffusers accelerate jiwer sentence-transformers soundfile

### Imports and the standard formats (PROVIDED)

Every station must read and write the **standard formats** so that stations from different teams can be chained on Game Day:

| Artifact | Format in Python | Format on disk |
|----------|------------------|----------------|
| Image | `PIL.Image`, RGB, 512 × 512 | PNG |
| Audio | 1-D `float32` NumPy array plus an integer sample rate | WAV, mono, 16,000 Hz, 16-bit |
| Text | `str`, at most 60 words | UTF-8 `.txt` |

In [ ]:
import os, json, time, glob, zipfile
import numpy as np
import torch
import matplotlib.pyplot as plt
import soundfile as sf
from PIL import Image, ImageDraw
from scipy.signal import resample_poly
from math import gcd

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
IMAGE_SIZE = 512      # standard image: 512 x 512 RGB
AUDIO_SR = 16000      # standard audio sample rate on disk
MAX_WORDS = 60        # standard text length limit
print("Running on", DEVICE)
if DEVICE == "cpu":
    print("No GPU found. In Colab: Runtime -> Change runtime type -> T4 GPU.")


def to_standard_image(image):
    """Return an RGB image center-cropped and resized to IMAGE_SIZE x IMAGE_SIZE."""
    image = image.convert("RGB")
    w, h = image.size
    side = min(w, h)
    left, top = (w - side) // 2, (h - side) // 2
    image = image.crop((left, top, left + side, top + side))
    return image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)


def limit_words(text, max_words=MAX_WORDS):
    """Collapse whitespace and keep at most max_words words."""
    words = str(text).split()
    return " ".join(words[:max_words])


def resample(audio, sr, target_sr=AUDIO_SR):
    """Resample a 1-D float audio array from sr to target_sr."""
    audio = np.asarray(audio, dtype=np.float32).reshape(-1)
    if sr == target_sr:
        return audio
    g = gcd(int(sr), int(target_sr))
    return resample_poly(audio, target_sr // g, int(sr) // g).astype(np.float32)


def make_test_image():
    """A simple synthetic scene used by check_stations(): sky, grass, sun, red ball."""
    img = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (110, 170, 235))
    d = ImageDraw.Draw(img)
    d.rectangle([0, 330, IMAGE_SIZE, IMAGE_SIZE], fill=(70, 160, 80))
    d.ellipse([360, 50, 460, 150], fill=(250, 215, 60))
    d.ellipse([200, 300, 300, 400], fill=(210, 40, 40))
    return img

### Load the starter models (PROVIDED)

These are the starter models from the project overview. Each loads once and is reused by your stations. You may swap any of them for another open model; if you do, update the matching station.

The first run downloads about 7 GB of model weights, which takes a few minutes.

In [ ]:
from transformers import (BlipProcessor, BlipForConditionalGeneration,
                          VitsModel, AutoTokenizer, pipeline)
from diffusers import AutoPipelineForText2Image

# Describe: image -> text
cap_proc = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
cap_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base").to(DEVICE)

# Speak: text -> audio
tts_tok = AutoTokenizer.from_pretrained("facebook/mms-tts-eng")
tts_model = VitsModel.from_pretrained("facebook/mms-tts-eng").to(DEVICE)

# Listen: audio -> text
asr = pipeline("automatic-speech-recognition", model="openai/whisper-base", device=DEVICE)

# Draw: text -> image
t2i = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=DTYPE).to(DEVICE)
t2i.set_progress_bar_config(disable=True)
print("Models loaded.")

## Part A: The Four Stations

Write each station as a function that follows the standard formats exactly. Keep them **deterministic where you can** (for example, use a fixed seed in `draw`), so your experiments are repeatable.

### TODO 1: `describe(image) -> str`

Caption an image with BLIP.

1. Convert the image with `cap_proc(images=image.convert("RGB"), return_tensors="pt")` and move the result to `DEVICE` with `.to(DEVICE)`.
2. Generate token IDs with `cap_model.generate(**inputs, max_new_tokens=40)` inside `with torch.no_grad():`.
3. Decode with `cap_proc.decode(ids[0], skip_special_tokens=True)`.
4. Return the caption passed through `limit_words()`.

In [ ]:
def describe(image):
    """Image (PIL) -> caption (str, at most MAX_WORDS words)."""
    # TODO 1
    raise NotImplementedError("TODO 1: describe")

### TODO 2: `speak(text) -> (audio, sample_rate)`

Turn text into speech with MMS-TTS.

1. Tokenize with `tts_tok(text, return_tensors="pt").to(DEVICE)`.
2. Run `tts_model(**inputs)` inside `torch.no_grad()`; the audio is in `.waveform[0]`.
3. Move it to the CPU as a NumPy `float32` array.
4. Return the array and `tts_model.config.sampling_rate` (MMS-TTS produces 16,000 Hz audio).

Edge case: if `text` is empty, return a short stretch of silence instead of crashing.

In [ ]:
def speak(text):
    """Text (str) -> (1-D float32 audio array, sample rate in Hz)."""
    # TODO 2
    raise NotImplementedError("TODO 2: speak")

### TODO 3: `listen(audio, sample_rate) -> str`

Transcribe speech with Whisper.

1. Whisper expects 16,000 Hz audio, so first call `resample(audio, sample_rate, 16000)`.
2. Call `asr({"raw": audio_16k, "sampling_rate": 16000}, generate_kwargs={"language": "en", "task": "transcribe"})`.
3. The transcript is in the result's `"text"` field. Return it stripped and passed through `limit_words()`.

In [ ]:
def listen(audio, sample_rate):
    """(audio array, sample rate) -> transcript (str)."""
    # TODO 3
    raise NotImplementedError("TODO 3: listen")

### TODO 4: `draw(text, seed=0) -> image`

Generate an image with SD-Turbo, which needs only **one** denoising step.

1. Make a seeded random generator: `torch.Generator(device=DEVICE).manual_seed(seed)`.
2. Call `t2i(prompt=text, num_inference_steps=1, guidance_scale=0.0, height=512, width=512, generator=g)`. SD-Turbo is trained without classifier-free guidance, so `guidance_scale` must be `0.0`.
3. Take `.images[0]` and return it passed through `to_standard_image()`.

In [ ]:
def draw(text, seed=0):
    """Text (str) -> 512 x 512 RGB image (PIL)."""
    # TODO 4
    raise NotImplementedError("TODO 4: draw")

### Check your stations (PROVIDED)

`check_stations()` runs every station on a test input and checks that it follows the standard formats. **All four must pass for Milestone 1.** Take a screenshot of the output for your submission.

In [ ]:
def check_stations(verbose=True):
    """Run each station once and check its output format. Returns True if all pass."""
    results = {}
    test_text = "a red ball on green grass under a blue sky"

    def run(name, fn, check):
        t0 = time.time()
        try:
            out = fn()
            problem = check(out)
        except NotImplementedError as e:
            out, problem = None, f"not implemented yet ({e})"
        except Exception as e:
            out, problem = None, f"error: {type(e).__name__}: {e}"
        results[name] = problem
        if verbose:
            status = "PASS" if problem is None else "FAIL"
            print(f"[{status}] {name:<8} {time.time() - t0:5.1f}s  {problem or ''}")
        return out

    def check_text(t):
        if not isinstance(t, str):
            return f"must return str, got {type(t).__name__}"
        if not t.strip():
            return "returned empty text"
        if len(t.split()) > MAX_WORDS:
            return f"more than {MAX_WORDS} words"
        return None

    def check_audio(out):
        if not (isinstance(out, tuple) and len(out) == 2):
            return "must return (audio, sample_rate)"
        a, sr = out
        if not isinstance(a, np.ndarray) or a.ndim != 1:
            return "audio must be a 1-D NumPy array"
        if a.dtype != np.float32:
            return f"audio must be float32, got {a.dtype}"
        if not isinstance(sr, (int, np.integer)) or sr <= 0:
            return "sample rate must be a positive int"
        if len(a) < sr * 0.2:
            return "audio is shorter than 0.2 seconds"
        if not np.isfinite(a).all():
            return "audio contains NaN or inf"
        return None

    def check_image(img):
        if not isinstance(img, Image.Image):
            return f"must return a PIL image, got {type(img).__name__}"
        if img.size != (IMAGE_SIZE, IMAGE_SIZE) or img.mode != "RGB":
            return f"must be {IMAGE_SIZE}x{IMAGE_SIZE} RGB, got {img.size} {img.mode}"
        return None

    caption = run("describe", lambda: describe(make_test_image()), check_text)
    spoken = run("speak", lambda: speak(caption or test_text), check_audio)
    if spoken is not None and results["speak"] is None:
        run("listen", lambda: listen(*spoken), check_text)
    else:
        results["listen"] = "skipped: listen() is tested on speak()'s audio, so fix speak() first"
        if verbose:
            print(f"[SKIP] listen          {results['listen']}")
    run("draw", lambda: draw(test_text), check_image)

    ok = all(p is None for p in results.values())
    if verbose:
        print("\nAll stations pass." if ok else "\nFix the failing stations above.")
    return ok

check_stations()

## Part B: The Game Loop

A **chain** is a list of rounds. Round 0 holds only the starting image. Every later round is a dictionary with exactly these keys:

| Key | Value |
|-----|-------|
| `"round"` | round number, starting at 1 |
| `"caption"` | text from `describe` on the previous round's image |
| `"audio"`, `"sr"` | output of `speak` on the caption |
| `"transcript"` | text from `listen` on that audio |
| `"image"` | image from `draw` on the transcript |
| `"seconds"` | how long the round took |

### TODO 5: `play_round(image, round_number, seed)`

Run the four stations in order on `image` and return one round dictionary. Use `seed + round_number` as the seed for `draw`, so each round gets a different but repeatable seed.

### TODO 6: `run_telephone(start_image, rounds=5, seed=0)`

Start the chain with `{"round": 0, "image": to_standard_image(start_image)}`, then call `play_round` on the latest image `rounds` times. Print each round's caption and transcript as you go, and return the chain.

In [ ]:
def play_round(image, round_number, seed=0):
    """Run Describe -> Speak -> Listen -> Draw once. Returns a round dict."""
    # TODO 5
    raise NotImplementedError("TODO 5: play_round")


def run_telephone(start_image, rounds=5, seed=0):
    """Play the game for `rounds` rounds. Returns the chain (list of round dicts)."""
    # TODO 6
    raise NotImplementedError("TODO 6: run_telephone")

### Save and view chains (PROVIDED)

`save_chain()` writes a chain to a folder using the standard file names, for example `round_01_caption.txt`, `round_01_speech.wav`, `round_01_transcript.txt`, and `round_01_image.png`. **Save every chain**: you will zip them for Milestone 1 and the final submission. In Colab, consider saving to Google Drive so results survive a disconnect.

In [ ]:
def save_text(text, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write(limit_words(text))


def save_audio(audio, sr, path):
    sf.write(path, resample(audio, sr, AUDIO_SR), AUDIO_SR, subtype="PCM_16")


def save_image(image, path):
    to_standard_image(image).save(path, format="PNG")


def save_chain(chain, folder, metrics=None):
    """Write every artifact in a chain (and optional metrics) to `folder`."""
    os.makedirs(folder, exist_ok=True)
    save_image(chain[0]["image"], os.path.join(folder, "round_00_image.png"))
    for step in chain[1:]:
        p = os.path.join(folder, f"round_{step['round']:02d}_")
        save_text(step["caption"], p + "caption.txt")
        save_audio(step["audio"], step["sr"], p + "speech.wav")
        save_text(step["transcript"], p + "transcript.txt")
        save_image(step["image"], p + "image.png")
    if metrics is not None:
        with open(os.path.join(folder, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)
    print("Saved", len(chain) - 1, "rounds to", folder)


def show_chain(chain, title=""):
    """Show every image in a chain in one row, captioned with the transcript that made it."""
    n = len(chain)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.6))
    axes = np.atleast_1d(axes)
    for ax, step in zip(axes, chain):
        ax.imshow(step["image"])
        ax.axis("off")
        label = "start" if step["round"] == 0 else f"R{step['round']}: {step['transcript']}"
        ax.set_title(label[:60], fontsize=8, wrap=True)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def zip_folder(folder, zip_path):
    """Zip a folder (for example, all your chains) for submission."""
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for path in glob.glob(os.path.join(folder, "**", "*"), recursive=True):
            if os.path.isfile(path):
                z.write(path, os.path.relpath(path, os.path.dirname(folder)))
    print("Wrote", zip_path)

Try one short chain on the test image. When this works, you are ready for Milestone 1: save the chain, zip it, and submit with your `check_stations()` screenshot.

In [ ]:
demo_chain = run_telephone(make_test_image(), rounds=2, seed=0)
show_chain(demo_chain, "Demo chain")
save_chain(demo_chain, "chains/demo")

## Part C: Measure the Drift

### Metric functions (PROVIDED)

These implement the metrics from the lesson *Chaining Models and What Gets Lost*:

- `wer_score(reference, hypothesis)`: word error rate after normalizing case and punctuation (0 is perfect).
- `text_similarity(a, b)`: cosine similarity of sentence embeddings (1 means the same meaning).
- `clip_image_similarity(img_a, img_b)`: cosine similarity of CLIP image embeddings (1 means identical).
- `clip_text_image_similarity(text, img)`: CLIP image–text similarity (a good match is often around 0.3).

In [ ]:
import jiwer
from sentence_transformers import SentenceTransformer
from transformers import CLIPModel, CLIPProcessor

text_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


def normalize_text(t):
    t = "".join(c if (c.isalnum() or c.isspace()) else " " for c in str(t).lower())
    return " ".join(t.split())


def wer_score(reference, hypothesis):
    ref, hyp = normalize_text(reference), normalize_text(hypothesis)
    if not ref:
        return 0.0 if not hyp else 1.0
    return float(jiwer.wer(ref, hyp))


def text_similarity(a, b):
    e = text_encoder.encode([str(a), str(b)], normalize_embeddings=True)
    return float(np.dot(e[0], e[1]))


def _unit(features):
    # transformers may return a tensor or an output object, depending on version
    if not torch.is_tensor(features):
        features = features.pooler_output
    return features / features.norm(dim=-1, keepdim=True)


@torch.no_grad()
def clip_image_embedding(image):
    inputs = clip_proc(images=image.convert("RGB"), return_tensors="pt").to(DEVICE)
    return _unit(clip_model.get_image_features(**inputs))


@torch.no_grad()
def clip_text_embedding(text):
    inputs = clip_proc(text=[str(text)], return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    return _unit(clip_model.get_text_features(**inputs))


def clip_image_similarity(img_a, img_b):
    return float((clip_image_embedding(img_a) @ clip_image_embedding(img_b).T).item())


def clip_text_image_similarity(text, image):
    return float((clip_text_embedding(text) @ clip_image_embedding(image).T).item())


print("WER example:", round(wer_score("a brown dog running through a field",
                                      "a brown dog running through the field"), 3))

### TODO 7: `chain_metrics(chain)`

Return one dictionary **per round** (rounds 1 and later) with these keys:

| Key | How to compute it |
|-----|-------------------|
| `"round"` | the round number |
| `"wer"` | `wer_score(caption, transcript)`: how much the speech link lost |
| `"caption_vs_round1"` | `text_similarity(caption, round-1 caption)`: how far the meaning drifted |
| `"image_vs_start"` | `clip_image_similarity(round image, starting image)`: how far the picture drifted |
| `"caption_matches_image"` | `clip_text_image_similarity(caption, image the caption describes)`: how good the Describe step was |

Note that each round's caption describes the *previous* round's image.

In [ ]:
def chain_metrics(chain):
    """Return a list of per-round metric dicts for rounds 1..N."""
    # TODO 7
    raise NotImplementedError("TODO 7: chain_metrics")

### Plot drift (PROVIDED)

`plot_drift(results)` takes a dictionary mapping each starting image's name to its `chain_metrics` output and draws one line per image for each metric.

In [ ]:
def plot_drift(results, keys=("image_vs_start", "caption_vs_round1", "wer")):
    titles = {"image_vs_start": "CLIP image similarity to start (higher = less drift)",
              "caption_vs_round1": "Caption similarity to round 1 (higher = less drift)",
              "wer": "Word error rate of the speech link (lower = better)",
              "caption_matches_image": "Caption-image CLIP match"}
    fig, axes = plt.subplots(1, len(keys), figsize=(5.5 * len(keys), 4))
    axes = np.atleast_1d(axes)
    for ax, key in zip(axes, keys):
        for name, rows in results.items():
            ax.plot([r["round"] for r in rows], [r[key] for r in rows], marker="o", label=name)
        ax.set_title(titles.get(key, key), fontsize=10)
        ax.set_xlabel("round")
        ax.grid(alpha=0.3)
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


demo_metrics = chain_metrics(demo_chain)
for row in demo_metrics:
    print({k: round(v, 3) if isinstance(v, float) else v for k, v in row.items()})

### TODO 8: Run the experiment

1. Put **at least 5 varied starting images** in a folder named `start_images/` (in Colab, use the file browser on the left, or mount Google Drive). Credit any image you did not create in your report.
2. For each image, run **at least 5 rounds**, compute `chain_metrics`, save the chain **with its metrics** to `chains/<image name>/`, and store the metrics in `results[<image name>]`.
3. Call `plot_drift(results)` and `show_chain()` for at least two chains you will discuss in your report.
4. Zip the `chains/` folder with `zip_folder("chains", "chains.zip")` for the final submission.

Use the same `seed` for every image so the comparison is fair.

In [ ]:
ROUNDS = 5
SEED = 0
results = {}

# TODO 8

## Part D: Extensions

Complete **two** extensions from the project overview (three for a team of three), one led by each partner:

1. **Model showdown:** swap one station's model and compare.
2. **Slow the drift:** design and test an intervention.
3. **The sound route:** add a MusicGen + CLAP branch.
4. **A noisy line:** add background noise before Listen.
5. **Your own idea** (approved at Milestone 0).

For each extension, re-run the **same starting images, rounds, and seed** as your Part C baseline, and compare with `plot_drift` or a table. Put each extension in its own section below, and explain the results in your report.

### TODO 9: Extension 1

In [ ]:
# TODO 9: Extension 1 (led by: ...)

### TODO 9: Extension 2

In [ ]:
# TODO 9: Extension 2 (led by: ...)

## Game Day Helper (PROVIDED)

On Game Day you run **one station** on a file you receive from the previous team and pass your output to the next team. `run_station_on_file()` reads the input in the standard format, runs your station, and writes the output in the standard format.

| Station | Input file | Output file |
|---------|-----------|-------------|
| `"describe"` | `.png` image | `.txt` text |
| `"speak"` | `.txt` text | `.wav` audio |
| `"listen"` | `.wav` audio | `.txt` text |
| `"draw"` | `.txt` text | `.png` image |

**Test it before Game Day** with the example below. A team whose files cannot be read breaks the chain for everyone after it.

In [ ]:
def load_text(path):
    with open(path, encoding="utf-8") as f:
        return limit_words(f.read())


def load_audio(path):
    audio, sr = sf.read(path, dtype="float32", always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    return audio.astype(np.float32), int(sr)


def run_station_on_file(station, in_path, out_path, seed=0):
    """Run one station on a standard-format file and write a standard-format file."""
    if station == "describe":
        save_text(describe(to_standard_image(Image.open(in_path))), out_path)
    elif station == "speak":
        audio, sr = speak(load_text(in_path))
        save_audio(audio, sr, out_path)
    elif station == "listen":
        save_text(listen(*load_audio(in_path)), out_path)
    elif station == "draw":
        save_image(draw(load_text(in_path), seed=seed), out_path)
    else:
        raise ValueError("station must be describe, speak, listen, or draw")
    print(f"{station}: {in_path} -> {out_path}")


# Round-trip test through all four stations using files only.
os.makedirs("gameday_test", exist_ok=True)
save_image(make_test_image(), "gameday_test/0_start.png")
run_station_on_file("describe", "gameday_test/0_start.png", "gameday_test/1_caption.txt")
run_station_on_file("speak", "gameday_test/1_caption.txt", "gameday_test/2_speech.wav")
run_station_on_file("listen", "gameday_test/2_speech.wav", "gameday_test/3_transcript.txt")
run_station_on_file("draw", "gameday_test/3_transcript.txt", "gameday_test/4_image.png")
print("Caption:   ", load_text("gameday_test/1_caption.txt"))
print("Transcript:", load_text("gameday_test/3_transcript.txt"))

## Submitting

See *Project 1: Teams, Milestones, and Submission* on the course website for deadlines and Gradescope links. Every submission is a **group submission**: one partner uploads and adds the other with **Add Group Member**.

**Milestone 1 (stations check-in):**
1. A screenshot of `check_stations()` with all four stations passing.
2. This notebook exported as a `.py` file.
3. Your demo chain zipped: `zip_folder("chains/demo", "demo_chain.zip")`.

**Final submission:**
1. This notebook exported as a `.py` file, named `lastname1_lastname2_project1.py`. Run every cell top to bottom first.
2. Your report as a PDF.
3. `chains.zip` from TODO 8, plus any extension chains.

**To export as `.py`:**
- **Google Colab:** File → Download → Download .py
- **VS Code:** the "..." menu at the top of the notebook → Export → Python Script